# **Perspektif Dönüşümler**

####**Bu derste şunları öğreneceğiz:**
1. OpenCV'nin getPerspectiveTransform'unu kullanma
2. Köşeleri almak ve perspektif Dönüşümünü otomatikleştirmek için findContours kullanma

Örnek bir uygulama; 
Taranmış bir belge (veya dikdörtgen nesnenin) perspektifini düzeltmek(yamuk görünen bir belgeyi dikdörtgen hale getirmek) 


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

In [ ]:
image = cv2.imread('../files/images/america22.jpg')

# Gri Tonlamaya Dönüştür
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
imshow('Eşiklemeden önce', gray,6)

_, th2 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
imshow('Eşiklemeden sonra', th2,6)

# findContours görüntüyü değiştirdiği için görüntünüzün bir kopyasını kullanın, örneğin edged.copy()
contours, hierarchy = cv2.findContours(th2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Tüm konturları çizin, bunun girdi görüntüsünün üzerine yazdığını unutmayın (yerinde işlem)
# Tümünü çizmek için 3. parametre olarak '-1' kullanın
cv2.drawContours(image, contours, -1, (0,255,0), thickness = 2)
imshow('Orijinal görüntü üzerine konturlar yerleştirildi', image)

print("Bulunan Kontur Sayısı = " + str(len(contours)))

### **Yukarıdaki konturumuzu approxPolyDP kullanarak sadece 4 noktaya uygulayın**

In [ ]:
# Konturları alana göre büyükten küçüğe doğru sıralayın 
# Konturların sıralanması, en büyük dörtgeni ilk sırada bulabilmek için yapılır.
#Eğer sıralama yapmazsan, döngü daha küçük ve alakasız konturları da test eder, hatta yanlış bir dörtgen bulabilir.
sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)

# konturlar üzerinde döngü
for cnt in sorted_contours:
	#konturu tahmin et
	perimeter = cv2.arcLength(cnt, True)
	approx = cv2.approxPolyDP(cnt, 0.05 * perimeter, True)
 
	if len(approx) == 4:  #len(approx) == 4 ise, bu kontur dört köşeli bir şekli (dikdörtgen veya yamuk) temsil eder.
		break             #bulunca döngüyü durdur

# Dört köşenin x, y koordinatları
print("Our 4 corner points are:")
print(approx)



### **Yukarıdan aşağıya görünümümüzü oluşturmak için getPerspectiveTransform ve warpPerspective kullanın**

Not: Noktaların sırasını anlamlı bir şekilde eşleştirdik

In [ ]:
# Burada elde edilen sıralama sol üst, sol alt, sağ alt, sağ üst şeklindedir
inputPts = np.float32(approx)  #inputPts: belgeden tespit edilen dört köşe noktasıdır.

outputPts = np.float32([[0,0],  #bu noktaların dönüştürüleceği hedef dikdörtgendir (500x800 piksel).
                       [0,800],
                       [500,800],
                       [500,0]])

# Dönüşüm Matrisimizi bulalım, M
M = cv2.getPerspectiveTransform(inputPts,outputPts) #cv2.getPerspectiveTransform dönüşüm matrisini hesaplar.

# Warp Perspective kullanarak M dönüşüm matrisini uygulayın
dst = cv2.warpPerspective(image, M, (500,800)) #cv2.warpPerspective ile orijinal görüntü, bu matris kullanılarak düzeltilir.

imshow("Perspective", dst) #Sonuçta belge düzgün bir dikdörtgen haline gelir.